In [1]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver

from langchain_core.messages import (
    HumanMessage,
    AIMessage,
    SystemMessage,
    trim_messages
)

from langchain_core.messages.utils import count_tokens_approximately

from langchain_ollama import ChatOllama

In [2]:
class State(MessagesState):
    summary: str

In [3]:
llm = ChatOllama(
    model="llama3.2",
    temperature=0,
    base_url="http://172.31.0.1:11434"
)

In [4]:
MAX_TOKENS = 100
RECENT_MESSAGES = 4

In [5]:
def summarize_messages(messages, previous_summary=""):

    conversation = ""

    if previous_summary:
        conversation += f"""
Previous summary:
{previous_summary}

"""

    for message in messages:
        conversation += f"{message.type}: {message.content}\n"

    prompt = f"""
You are maintaining memory for a conversation.

Create a concise factual memory summary.

IMPORTANT:
Preserve personal information explicitly given by the user,
such as their name, preferences, goals, projects, and important facts.

Preserve important information needed to answer future questions.

Do NOT write essays.
Do NOT answer questions.
Do NOT invent information.

Conversation:
{conversation}

Return ONLY the memory summary.
"""

    response = llm.invoke(
        [HumanMessage(content=prompt)]
    )

    return response.content

In [6]:
def call_model(state: State):

    messages = state["messages"]
    previous_summary = state.get("summary", "")

    token_count = count_tokens_approximately(messages)

    print("Current token count:", token_count)

    # -----------------------------------------
    # Conversation is still below 100 tokens
    # -----------------------------------------

    if token_count <= MAX_TOKENS:

        context = messages
        summary = previous_summary

    # -----------------------------------------
    # Conversation exceeded 100 tokens
    # -----------------------------------------

    else:

        print("Token limit exceeded!")
        print("Creating summary...")

        # Keep the latest few messages
        recent_messages = messages[-RECENT_MESSAGES:]

        # Everything before recent messages
        old_messages = messages[:-RECENT_MESSAGES]

        # Summarize old conversation
        summary = summarize_messages(
            old_messages,
            previous_summary
        )

        print("Summary created!")
        print("SUMMARY:", summary)

        # IMPORTANT:
        # Keep summary + recent messages.
        # Do NOT run trim_messages(max_tokens=100)
        # over the summary itself because it can remove it.

        context = [
            SystemMessage(
                content=f"""
You have memory from the earlier conversation.

MEMORY:
{summary}

Use this memory when answering the user.
"""
            )
        ] + recent_messages

    # -----------------------------------------
    # Main model
    # -----------------------------------------

    response = llm.invoke(context)

    return {
        "messages": [response],
        "summary": summary
    }

In [7]:
builder = StateGraph(State)

builder.add_node("call_model", call_model)

builder.add_edge(START, "call_model")
builder.add_edge("call_model", END)

In [8]:
memory = InMemorySaver()

graph = builder.compile(
    checkpointer=memory
)

In [9]:
config = {
    "configurable": {
        "thread_id": "2"
    }
}

In [18]:
response = graph.invoke(
    {
        "messages": [
            HumanMessage(
                content="my hobies are??"
            )
        ]
    },
    config
)

print(response["messages"][-1].content)

Current token count: 1034
Token limit exceeded!
Creating summary...
Summary created!
SUMMARY: Memory Summary:

- Name: Sanjay
- Request 1: Write a 100-word essay on the benefits of exercise.
- Request 2: Write a 100-word essay on AI augmentation of human intelligence (first request)
- Personal preference: Favorite color is blue
- Hobbies: Enjoy hiking in the mountains, playing chess
You enjoy hiking in the mountains and playing chess as one of your hobbies, right?


In [ ]:
state = graph.get_state(config)

print("========== SUMMARY ==========")
print(state.values.get("summary", "No summary yet."))

print("\n========== MESSAGES ==========")

for message in state.values["messages"]:
    print(f"{message.type}: {message.content}")